# Guide du fil rouge : ce qui est à faire, et ce qu'il ne faut pas faire

Ce notebook ne contient pas le projet : le projet est dans `src/credit`. Il sert à vous guider, étape par étape, et à vous montrer comment on appelle le code du dépôt depuis un notebook. **Un notebook qui recopie `src/` est un notebook qui ment** : le code qu'on présente n'est plus celui qu'on exécute.

Lancez les cellules dans l'ordre. Chaque section correspond à une séance du cours et dit ce que vous devez rendre.

In [ ]:
# On importe le code du dépôt. Rien n'est recopié ici.
import sys, json
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

from credit.config import load_config
from credit.data import load_raw, TARGET
from credit.split import load_fold, make_split, fold_hash
from credit.features import split_xy, feature_columns, make_pipeline
from credit.model import naive, baseline, load_model
from credit.evaluate import metrics, cost_curve, best_threshold, calibration_table

cfg = load_config()   # se résout depuis la racine du dépôt, quel que soit le dossier courant
cfg["seed"], cfg["costs"], cfg["threshold"]

## Étape 1 (séance 1) : cadrer avant de modéliser

À faire avant d'écrire une ligne de modèle :

1. Lire `docs/cahier-des-charges.md` : la décision servie, la cible, la métrique, les données, les garde-fous, les livrables. Six lignes.
2. Ouvrir `data/README.md` : d'où viennent les données, ce que veut dire chaque colonne.
3. Dessiner la carte des fuites : quelle colonne pourrait contenir une information postérieure à la cible ? Ici aucune, mais sur votre sujet ce sera la première question.

**Attention à ne pas faire** : regarder les corrélations avec la cible sur le jeu entier avant la coupe. Ce que vous apprenez là contamine la validation.

In [ ]:
raw = load_raw()          # télécharge si besoin, vérifie le contrat de schéma
print(raw.shape)
print("taux de défaut :", round(raw[TARGET].mean(), 4))
raw.head()

## Étape 2 (séance 1) : la coupe figée

La coupe se fait **une fois**, avec la graine de `config.yaml`, et ses empreintes sont figées par un test. Le pli de test ne se lit qu'au jalon 2, une seule fois, avec `make compare`.

**Attention à ne pas faire** : relancer la coupe avec une autre graine « pour voir ». Si vous changez la coupe, vous l'écrivez dans `DECISIONS.md` et vous mettez le test à jour.

In [ ]:
folds = make_split(raw, cfg)
for name, df in folds.items():
    print(f"{name:11s} {len(df):6d} lignes, taux {df[TARGET].mean():.4f}, empreinte {fold_hash(df)[:12]}")

## Étape 3 (séances 1 et 2) : le modèle naïf, puis la logistique

Le modèle naïf prédit le taux de base à tout le monde. Sa log-loss est l'entropie de la cible, 0,528 nats : **c'est le plancher**. Un modèle qui ne le bat pas nettement est cassé ou fuit.

La logistique est un `Pipeline` : la standardisation apprend ses moyennes sur le pli d'entraînement, jamais sur le jeu entier. Puis une calibration de Platt sur la validation.

**Attention à ne pas faire** : `StandardScaler().fit(X_tout)` avant la coupe. C'est la fuite la plus courante du cours, et un test la détecterait.

In [ ]:
train = load_fold("train"); val = load_fold("validation")
X_tr, y_tr = split_xy(train, cfg)
X_val, y_val = split_xy(val, cfg)
print("variables :", feature_columns(cfg))
assert "SEX" not in X_tr.columns, "SEX est un garde-fou"

m_naive = naive(cfg)
m_base = baseline(cfg)
p_naive = m_naive.predict_proba(X_val)[:, 1]
p_base = m_base.predict_proba(X_val)[:, 1]
c_fp, c_fn, t = cfg["costs"]["c_fp"], cfg["costs"]["c_fn"], cfg["threshold"]
for name, p in (("naïf", p_naive), ("logistique", p_base)):
    m = metrics(y_val, p, t, c_fp, c_fn)
    print(f"{name:11s} AUC {m['auc']}  log-loss {m['log_loss']}  Brier {m['brier']}  coût/dossier {m['cout_attendu_par_dossier']} €")

## Étape 4 (séance 4) : le seuil est une décision, pas une sortie du modèle

Avec des probabilités calibrées, le seuil qui minimise le coût attendu vaut `c_fp / (c_fp + c_fn)`. On trace la courbe de coût C(t) et on regarde où elle est plate : c'est là que le modèle n'aide pas.

**Attention** : le seuil se choisit sur la validation, s'écrit dans `config.yaml`, se justifie dans `DECISIONS.md`. Jamais dans un script de batch, jamais dans l'interface.

In [ ]:
curve = cost_curve(y_val, p_base, c_fp, c_fn)
print("seuil d'indifférence :", round(c_fp / (c_fp + c_fn), 4))
print("argmin empirique     :", best_threshold(y_val, p_base, c_fp, c_fn))
ax = curve.plot(x="t", y="cost", legend=False, title="Coût attendu par dossier selon le seuil")
ax.axvline(t, linestyle="--"); ax.set_ylabel("euros")

In [ ]:
# La calibration : par tranche de probabilité prédite, la fréquence observée. Calibré = les deux se suivent.
import pandas as pd
pd.DataFrame(calibration_table(y_val, p_base))

## Étape 5 (jalon 1, 20 novembre) : ce que vous rendez

- `make reproduce` tourne chez une autre équipe et donne les mêmes chiffres.
- Les cinq tests et la CI sont verts.
- `ABLATION.md` : prédictions **pré-enregistrées dans une issue**, puis calculées.
- `DECISIONS.md` : dix entrées datées.
- Deux notebooks « from scratch » dans ce dossier : les moindres carrés et la logistique réécrits à la main (NumPy), qui retrouvent les coefficients de scikit-learn à 1e-6.

**Attention** : une ablation se fait en réentraînant sans le groupe de colonnes, pas en les mettant à zéro.

## Étape 6 (jalon 2, 5 février) : le modèle lourd et la comparaison honnête

`credit.model.heavy()` vous dit quoi faire et refuse de tourner tant qu'il n'est pas écrit. Les hyperparamètres se cherchent par validation croisée **sur le pli d'entraînement seulement** ; la validation sert à calibrer et au seuil ; le test se lit une fois, après avoir écrit la « Règle du champion » dans `DECISIONS.md`.

**Attention** : comparer deux AUC sans intervalle n'est pas un résultat. Test apparié corrigé (Nadeau et Bengio) sur les plis, voir la séance 6.

In [ ]:
# make compare refusera de tourner tant que heavy() n'est pas écrit et que la règle du champion manque.
# !make compare

## Étape 7 (jalon 3, 11 mars) : livrer, surveiller, défendre

`MONITORING.md` (PSI, dérive de la cible, alarmes), `MEMO_CLIENT.md` (une page en euros), la fiche de modèle v2 avec l'audit d'équité sur `MARRIAGE`, l'interface `app/app.py`, et la soutenance : quatorze minutes, puis des questions à chaque membre, sur n'importe quel fichier que vous avez signé.

L'IA peut écrire du code ici. À l'oral, vous devez pouvoir expliquer chaque ligne.